# Hurdle T-learner baseline for uplift@10

Baseline for the Magnit uplift case:

- CatBoost is used as the base learner.
- Treatment/control are modeled separately as a T-learner.
- Because `rec_spend` has many zeros, each arm is a hurdle model:
  - probability of non-zero spend;
  - expected amount conditional on non-zero spend.
- Validation optimizes uplift ranking: `uplift@10`, not RMSE.
- Main validation: stratified split/CV by `treatment_flg + communication_type`.
- Extra stress-test: holdout by the largest `user_id` values, because public test ids continue after train ids.

In [ ]:
import gc
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

DATA_DIR = Path("кей")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"
PREDICTIONS_PATH = Path("predictions.csv")

ID_COL = "user_id"
TREATMENT_COL = "treatment_flg"
TARGET_COL = "rec_spend"

TOP_K = 0.10
BOOTSTRAP_ITERS = 200
# Lower boundary of a central 80% bootstrap interval is the 10th percentile.
BOOTSTRAP_LOWER_Q = 0.10

# For a quick smoke run lower these values. For the final run increase iterations.
N_SPLITS = 5
VALID_SIZE = 0.25
CATBOOST_ITERATIONS = 450
CATBOOST_DEPTH = 6
CATBOOST_LR = 0.05
# Regularization is here mostly to reduce overfitting/noisy uplift, not because CatBoost cannot handle collinearity.
CATBOOST_L2_LEAF_REG = 5
CATBOOST_RANDOM_STRENGTH = 1
CATBOOST_BAGGING_TEMPERATURE = 1
CATBOOST_RSM = 0.85
THREAD_COUNT = 4

RUN_SINGLE_HOLDOUT = True
RUN_FULL_CV = True
RUN_USER_ID_STRESS_TEST = True
FIT_FINAL_MODEL = True

## Load data

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("train user_id range:", (train[ID_COL].min(), train[ID_COL].max()))
print("test user_id range :", (test[ID_COL].min(), test[ID_COL].max()))
print("user_id overlap:", len(set(train[ID_COL]) & set(test[ID_COL])))

display(train.head())

In [ ]:
# Test has exactly the feature columns that must be used at inference time.
FEATURES = [c for c in test.columns if c != ID_COL]
CAT_FEATURES = ["communication_type"]

assert ID_COL not in FEATURES
assert TREATMENT_COL not in FEATURES
assert TARGET_COL not in FEATURES
assert set(FEATURES).issubset(train.columns)

STRATA = train[TREATMENT_COL].astype(str) + "_" + train["communication_type"].astype(str)

print("n_features:", len(FEATURES))
print("cat_features:", CAT_FEATURES)
print("\nstrata counts:")
print(STRATA.value_counts().sort_index())

## Quick EDA checks

These are the key checks for this task: treatment balance, communication balance, and the zero-heavy target distribution.

In [ ]:
target_summary = pd.Series(
    {
        "mean": train[TARGET_COL].mean(),
        "std": train[TARGET_COL].std(),
        "zero_share": (train[TARGET_COL] == 0).mean(),
        "p50": train[TARGET_COL].quantile(0.50),
        "p90": train[TARGET_COL].quantile(0.90),
        "p95": train[TARGET_COL].quantile(0.95),
        "p99": train[TARGET_COL].quantile(0.99),
        "max": train[TARGET_COL].max(),
    }
)

print("Treatment balance:")
display(train[TREATMENT_COL].value_counts(normalize=True).rename("share").to_frame())

print("Communication x treatment balance:")
display(pd.crosstab(train["communication_type"], train[TREATMENT_COL], normalize="index"))

print("Target summary:")
display(target_summary.to_frame("value"))

print("Target by treatment:")
display(
    train.groupby(TREATMENT_COL)[TARGET_COL]
    .agg(
        n="size",
        mean="mean",
        zero_share=lambda s: (s == 0).mean(),
        p90=lambda s: s.quantile(0.90),
        p95=lambda s: s.quantile(0.95),
        p99=lambda s: s.quantile(0.99),
        max="max",
    )
)

print("Raw uplift by communication_type:")
raw_by_comm = []
for communication_type, part in train.groupby("communication_type"):
    mean_t = part.loc[part[TREATMENT_COL] == 1, TARGET_COL].mean()
    mean_c = part.loc[part[TREATMENT_COL] == 0, TARGET_COL].mean()
    raw_by_comm.append((communication_type, mean_t, mean_c, mean_t - mean_c))
display(pd.DataFrame(raw_by_comm, columns=["communication_type", "mean_treat", "mean_control", "raw_uplift"]))

## Uplift metrics

`uplift@10` is computed on the top 10% users sorted by predicted uplift score:

`mean(rec_spend | treatment=1, top10) - mean(rec_spend | treatment=0, top10)`.

The bootstrap below resamples only the selected top-K segment. This matches the business question: how stable is the measured uplift among users we would actually target?

In [ ]:
def uplift_at_k(y_true, treatment, score, k=TOP_K):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)

    assert len(y_true) == len(treatment) == len(score)

    n_top = max(1, int(len(score) * k))
    # Stable sort makes results reproducible when scores tie.
    top_idx = np.argsort(-score, kind="mergesort")[:n_top]
    top_y = y_true[top_idx]
    top_w = treatment[top_idx]

    treat_mask = top_w == 1
    control_mask = top_w == 0
    if treat_mask.sum() == 0 or control_mask.sum() == 0:
        uplift = np.nan
    else:
        uplift = top_y[treat_mask].mean() - top_y[control_mask].mean()

    return {
        "uplift_at_k": float(uplift),
        "n_total": int(len(score)),
        "n_top": int(n_top),
        "top_treatment_n": int(treat_mask.sum()),
        "top_control_n": int(control_mask.sum()),
        "top_target_mean": float(top_y.mean()),
        "top_score_mean": float(score[top_idx].mean()),
    }


def bootstrap_uplift_at_k(
    y_true,
    treatment,
    score,
    k=TOP_K,
    n_bootstrap=BOOTSTRAP_ITERS,
    lower_q=BOOTSTRAP_LOWER_Q,
    random_state=RANDOM_STATE,
):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)

    base = uplift_at_k(y_true, treatment, score, k=k)
    n_top = base["n_top"]
    top_idx = np.argsort(-score, kind="mergesort")[:n_top]

    top_y = y_true[top_idx]
    top_w = treatment[top_idx]

    rng = np.random.default_rng(random_state)
    boot_values = []
    for _ in range(n_bootstrap):
        sample_idx = rng.integers(0, n_top, size=n_top)
        sample_y = top_y[sample_idx]
        sample_w = top_w[sample_idx]
        treat_mask = sample_w == 1
        control_mask = sample_w == 0
        if treat_mask.sum() == 0 or control_mask.sum() == 0:
            continue
        boot_values.append(sample_y[treat_mask].mean() - sample_y[control_mask].mean())

    boot_values = np.asarray(boot_values, dtype=float)
    result = dict(base)
    result.update(
        {
            "bootstrap_mean": float(np.mean(boot_values)),
            "bootstrap_std": float(np.std(boot_values)),
            "bootstrap_lower_80": float(np.quantile(boot_values, lower_q)),
            "bootstrap_upper_80": float(np.quantile(boot_values, 1 - lower_q)),
            "bootstrap_iters_used": int(len(boot_values)),
        }
    )
    return result

## Hurdle T-learner

For each arm `w in {0, 1}`:

`E[rec_spend | X, w] = P(rec_spend > 0 | X, w) * E[rec_spend | rec_spend > 0, X, w]`

Final score:

`uplift_score = E[rec_spend | X, treatment=1] - E[rec_spend | X, treatment=0]`

In [ ]:
def make_prob_params(seed=RANDOM_STATE, verbose=False):
    return {
        "iterations": CATBOOST_ITERATIONS,
        "depth": CATBOOST_DEPTH,
        "learning_rate": CATBOOST_LR,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "l2_leaf_reg": CATBOOST_L2_LEAF_REG,
        "random_strength": CATBOOST_RANDOM_STRENGTH,
        "bagging_temperature": CATBOOST_BAGGING_TEMPERATURE,
        "rsm": CATBOOST_RSM,
        "random_seed": seed,
        "thread_count": THREAD_COUNT,
        "verbose": verbose,
        "allow_writing_files": False,
    }


def make_amount_params(seed=RANDOM_STATE, verbose=False):
    return {
        "iterations": CATBOOST_ITERATIONS,
        "depth": CATBOOST_DEPTH,
        "learning_rate": CATBOOST_LR,
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "l2_leaf_reg": CATBOOST_L2_LEAF_REG,
        "random_strength": CATBOOST_RANDOM_STRENGTH,
        "bagging_temperature": CATBOOST_BAGGING_TEMPERATURE,
        "rsm": CATBOOST_RSM,
        "random_seed": seed,
        "thread_count": THREAD_COUNT,
        "verbose": verbose,
        "allow_writing_files": False,
    }


def fit_hurdle_t_learner(df, features=FEATURES, cat_features=CAT_FEATURES, seed=RANDOM_STATE, verbose=False):
    models = {}

    for arm in [0, 1]:
        arm_mask = df[TREATMENT_COL].eq(arm)
        x_arm = df.loc[arm_mask, features]
        y_arm = df.loc[arm_mask, TARGET_COL].astype(float)
        y_nonzero = y_arm.gt(0).astype(int)

        print(
            f"arm={arm}: rows={len(y_arm):,}, nonzero={int(y_nonzero.sum()):,}, "
            f"nonzero_share={y_nonzero.mean():.4f}"
        )

        prob_model = CatBoostClassifier(**make_prob_params(seed=seed + arm, verbose=verbose))
        prob_model.fit(x_arm, y_nonzero, cat_features=cat_features)

        positive_mask = y_arm.gt(0)
        amount_model = CatBoostRegressor(**make_amount_params(seed=seed + 10 + arm, verbose=verbose))
        amount_model.fit(
            x_arm.loc[positive_mask],
            np.log1p(y_arm.loc[positive_mask]),
            cat_features=cat_features,
        )

        models[arm] = {
            "prob_model": prob_model,
            "amount_model": amount_model,
            "n_rows": int(len(y_arm)),
            "n_positive": int(positive_mask.sum()),
        }

    return models


def predict_expected_spend(models, x, arm):
    prob = models[arm]["prob_model"].predict_proba(x)[:, 1]
    amount_log = models[arm]["amount_model"].predict(x)
    amount = np.expm1(amount_log)
    amount = np.clip(amount, 0, None)
    return prob * amount


def predict_uplift(models, x):
    expected_treat = predict_expected_spend(models, x, arm=1)
    expected_control = predict_expected_spend(models, x, arm=0)
    return expected_treat - expected_control

## Single stratified holdout

Fast first validation. Split is stratified by `treatment_flg + communication_type`.

In [ ]:
if RUN_SINGLE_HOLDOUT:
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=VALID_SIZE,
        random_state=RANDOM_STATE,
    )
    tr_idx, va_idx = next(splitter.split(train, STRATA))

    train_part = train.iloc[tr_idx].reset_index(drop=True)
    valid_part = train.iloc[va_idx].reset_index(drop=True)

    print("train_part:", train_part.shape)
    print("valid_part:", valid_part.shape)
    print("\nvalid strata:")
    print((valid_part[TREATMENT_COL].astype(str) + "_" + valid_part["communication_type"].astype(str)).value_counts().sort_index())

    t0 = time.time()
    holdout_models = fit_hurdle_t_learner(train_part, seed=RANDOM_STATE, verbose=False)
    holdout_scores = predict_uplift(holdout_models, valid_part[FEATURES])
    holdout_metrics = bootstrap_uplift_at_k(
        valid_part[TARGET_COL].values,
        valid_part[TREATMENT_COL].values,
        holdout_scores,
        random_state=RANDOM_STATE,
    )
    print(f"elapsed: {(time.time() - t0) / 60:.1f} min")
    display(pd.Series(holdout_metrics).to_frame("holdout"))

    del train_part, valid_part
    gc.collect()

## StratifiedKFold validation

This is the main offline validation. The final OOF score is evaluated once on all out-of-fold predictions.

In [ ]:
def run_stratified_cv(df, n_splits=N_SPLITS, seed=RANDOM_STATE):
    strata = df[TREATMENT_COL].astype(str) + "_" + df["communication_type"].astype(str)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_scores = np.zeros(len(df), dtype=float)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(df, strata), start=1):
        print("=" * 80)
        print(f"fold {fold}/{n_splits}")
        train_part = df.iloc[tr_idx].reset_index(drop=True)
        valid_part = df.iloc[va_idx].reset_index(drop=True)

        t0 = time.time()
        models = fit_hurdle_t_learner(train_part, seed=seed + 100 * fold, verbose=False)
        scores = predict_uplift(models, valid_part[FEATURES])

        fold_metrics = bootstrap_uplift_at_k(
            valid_part[TARGET_COL].values,
            valid_part[TREATMENT_COL].values,
            scores,
            random_state=seed + fold,
        )
        fold_metrics["fold"] = fold
        fold_metrics["elapsed_min"] = (time.time() - t0) / 60
        fold_rows.append(fold_metrics)

        oof_scores[va_idx] = scores
        display(pd.Series(fold_metrics).to_frame(f"fold_{fold}"))

        del train_part, valid_part, models, scores
        gc.collect()

    fold_metrics_df = pd.DataFrame(fold_rows)
    oof_metrics = bootstrap_uplift_at_k(
        df[TARGET_COL].values,
        df[TREATMENT_COL].values,
        oof_scores,
        random_state=seed + 999,
    )

    return oof_scores, fold_metrics_df, oof_metrics


if RUN_FULL_CV:
    cv_oof_scores, cv_fold_metrics, cv_oof_metrics = run_stratified_cv(train)

    print("Fold metrics:")
    display(cv_fold_metrics)

    print("OOF metrics:")
    display(pd.Series(cv_oof_metrics).to_frame("oof"))

## Stress-test: upper `user_id` holdout

This is not time-series CV. It is only a distribution/generalization stress-test, because `test.user_id` starts after `train.user_id`.

In [ ]:
def make_upper_user_id_holdout(df, valid_size=VALID_SIZE):
    ordered_idx = df.sort_values(ID_COL).index.to_numpy()
    cut = int(len(ordered_idx) * (1 - valid_size))
    train_idx = ordered_idx[:cut]
    valid_idx = ordered_idx[cut:]
    return train_idx, valid_idx


if RUN_USER_ID_STRESS_TEST:
    tr_idx, va_idx = make_upper_user_id_holdout(train, valid_size=VALID_SIZE)
    train_part = train.loc[tr_idx].reset_index(drop=True)
    valid_part = train.loc[va_idx].reset_index(drop=True)

    print("train_part user_id range:", (train_part[ID_COL].min(), train_part[ID_COL].max()))
    print("valid_part user_id range:", (valid_part[ID_COL].min(), valid_part[ID_COL].max()))
    print("\nvalid treatment balance:")
    print(valid_part[TREATMENT_COL].value_counts(normalize=True))
    print("\nvalid communication balance:")
    print(valid_part["communication_type"].value_counts(normalize=True))

    t0 = time.time()
    stress_models = fit_hurdle_t_learner(train_part, seed=RANDOM_STATE + 2026, verbose=False)
    stress_scores = predict_uplift(stress_models, valid_part[FEATURES])
    stress_metrics = bootstrap_uplift_at_k(
        valid_part[TARGET_COL].values,
        valid_part[TREATMENT_COL].values,
        stress_scores,
        random_state=RANDOM_STATE + 2026,
    )
    print(f"elapsed: {(time.time() - t0) / 60:.1f} min")
    display(pd.Series(stress_metrics).to_frame("upper_user_id_holdout"))

    del train_part, valid_part
    gc.collect()

## Final training and `predictions.csv`

After choosing parameters, fit on the full train set and score the hidden test features.

In [ ]:
if FIT_FINAL_MODEL:
    t0 = time.time()
    final_models = fit_hurdle_t_learner(train, seed=RANDOM_STATE + 777, verbose=False)
    test_scores = predict_uplift(final_models, test[FEATURES])

    predictions = pd.DataFrame(
        {
            ID_COL: test[ID_COL].values,
            "UPLIFT_SCORE": test_scores,
        }
    )

    assert len(predictions) == len(test)
    assert predictions[ID_COL].tolist() == test[ID_COL].tolist()
    assert predictions["UPLIFT_SCORE"].notna().all()
    assert np.isfinite(predictions["UPLIFT_SCORE"]).all()

    predictions.to_csv(PREDICTIONS_PATH, index=False)

    print(f"saved: {PREDICTIONS_PATH.resolve()}")
    print(f"elapsed: {(time.time() - t0) / 60:.1f} min")
    display(predictions.head())
    display(predictions["UPLIFT_SCORE"].describe().to_frame("UPLIFT_SCORE"))

## Optional: simple rank ensemble

If several uplift models are added later, prefer rank averaging over raw-score averaging. Raw scores from different meta-learners often have incompatible scales, while the leaderboard metric depends only on ordering.

In [ ]:
def rank_average(*score_arrays):
    ranks = []
    for scores in score_arrays:
        scores = np.asarray(scores)
        # Higher score -> higher rank value.
        ranks.append(pd.Series(scores).rank(method="average", ascending=True).to_numpy())
    return np.mean(ranks, axis=0)